# Temporal Fusion Transformer Training
Automated TFT configuration combining exploratory data parsing rules and the PyTorch TFTv2 architecture.

All variables will be evaluated intrinsically by the model's self-contained `VariableSelectionNetwork` so manual PCA and dimensionality drop is ignored.

In [ ]:
import os, sys, random, warnings, time, logging
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()} - {torch.cuda.get_device_name(0) if torch.cuda.is_available() else ''}")

# Checkpointing & Logging Setup
OUTPUT_DIR = Path("checkpoints_04")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = OUTPUT_DIR / f"training_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.FileHandler(LOG_FILE, encoding="utf-8"),
        logging.StreamHandler(),
    ],
    force=True
)
log = logging.getLogger(__name__)
log.info("Started TFT Pipeline Configuration")

## Configuration Space
All hyperparameters are stored here. Sequence length assumes `4` since financial market sequences change fast (tuned against overfitting).

In [ ]:
# TFT Hyperparameters
SEQ_LEN        = 4           
BATCH_SIZE     = 1024        
MAX_TRAIN_SEQ  = None           
MAX_VAL_SEQ    = 500_000
TOP_K_FEATURES = 30
HIDDEN_DIM     = 48          
NUM_HEADS      = 8
DROPOUT        = 0.3         
EMBED_DIM      = 8           
LR             = 1e-3        
WEIGHT_DECAY   = 1e-3        
EPOCHS         = 40          
PATIENCE       = 5
CLIP_NORM      = 1.0

## Data Fetching
Fetching temporal market elements via `polars` into `pandas` DataFrame

In [ ]:
log.info("Loading Data...")
DATA_DIR   = Path(".")
# Local Paths explicitly
TRAIN_PATH = DATA_DIR / "data/train.parquet"
TEST_PATH  = DATA_DIR / "data/test.parquet"

# Using pandas directly or polars if available
train_pl = pl.scan_parquet(str(TRAIN_PATH)).collect()
test_pl  = pl.scan_parquet(str(TEST_PATH)).collect()
train_pd = train_pl.to_pandas()
test_pd  = test_pl.to_pandas()
del train_pl, test_pl

log.info(f"Train Shape: {train_pd.shape} | Test Shape: {test_pd.shape}")

# Define Columns
TARGET_COL  = "y_target"
WEIGHT_COL  = "weight"
TIME_COL    = "ts_index"
ID_COL      = "id"
HORIZON_COL = "horizon"
GROUP_COLS  = [c for c in ["code", "sub_code", "sub_category"] if c in train_pd.columns]

sort_cols = GROUP_COLS + [TIME_COL]
train_pd  = train_pd.sort_values(sort_cols)
test_pd   = test_pd.sort_values(sort_cols)

feature_cols_all = [c for c in train_pd.columns if c not in [TARGET_COL, WEIGHT_COL, ID_COL]]
numeric_cols_all = [c for c in feature_cols_all if pd.api.types.is_numeric_dtype(train_pd[c])]
cat_feature_cols = [c for c in feature_cols_all if c not in numeric_cols_all]

## Exploratory Validation Splits and Engineering
Target value normalization against their exact horizons. Uses a logged weight formula matching `tft_model_v2.py` logic.

In [ ]:
log.info("Performing chronological split and scaling...")
unique_times = np.sort(train_pd[TIME_COL].unique())
split_time   = unique_times[int(len(unique_times) * 0.8)]

train_df = train_pd.loc[train_pd[TIME_COL] <  split_time].copy()
val_df   = train_pd.loc[train_pd[TIME_COL] >= split_time].copy()
del train_pd

# Use raw targets without standard scaling (to match evaluation metric)
# Use raw weights (clipping strictly negative weights if any exist)
train_df[WEIGHT_COL] = train_df[WEIGHT_COL].clip(lower=0).values
val_df[WEIGHT_COL]   = val_df[WEIGHT_COL].clip(lower=0).values

log.info(f"Selecting Top {TOP_K_FEATURES} Features based on weighted absolute correlation...")
def weighted_abs_corr(x, y, w):
    mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(w)
    if mask.sum() < 10: return 0.0
    x, y, w = x[mask], y[mask], w[mask]
    w_sum = w.sum()
    if w_sum <= 0: return 0.0
    x_mean = np.sum(w * x) / w_sum
    y_mean = np.sum(w * y) / w_sum
    x_center = x - x_mean
    y_center = y - y_mean
    cov = np.sum(w * x_center * y_center) / w_sum
    x_var = np.sum(w * np.square(x_center)) / w_sum
    y_var = np.sum(w * np.square(y_center)) / w_sum
    if x_var <= 0 or y_var <= 0: return 0.0
    return float(abs(cov / np.sqrt(x_var * y_var)))

# Calculate correlation on train_df subset to speed up execution
scores = []
sample_size = min(len(train_df), 800_000)
# taking the most recent sample size since markets drift temporarily
sample_df = train_df.sort_values(TIME_COL).tail(sample_size)
y_vals = sample_df[TARGET_COL].values
w_vals = sample_df[WEIGHT_COL].values

for col in numeric_cols_all:
    c = weighted_abs_corr(sample_df[col].values, y_vals, w_vals)
    scores.append((col, c))
    
scores.sort(key=lambda x: x[1], reverse=True)
top_numeric_cols = [x[0] for x in scores[:TOP_K_FEATURES]]
numeric_cols_all = top_numeric_cols
log.info(f"Selected Top Features: {numeric_cols_all}")

log.info("Handling Missing Values via Imputation & Scaling...")
# Impute numericals with median
num_fill = {c: train_df[c].median() for c in numeric_cols_all}
for c, m in num_fill.items():
    train_df[c] = train_df[c].fillna(m)
    val_df[c]   = val_df[c].fillna(m)
    if c in test_pd.columns: test_pd[c] = test_pd[c].fillna(m)

# Scale Numericals
scaler = StandardScaler()
train_df[numeric_cols_all] = scaler.fit_transform(train_df[numeric_cols_all])
val_df[numeric_cols_all]   = scaler.transform(val_df[numeric_cols_all])
if set(numeric_cols_all).issubset(test_pd.columns):
    test_pd[numeric_cols_all] = scaler.transform(test_pd[numeric_cols_all])

# Categorical Encoding
cat_maps = {}
cat_vocab_sizes = {}
for c in cat_feature_cols:
    vals  = train_df[c].astype(str).fillna("__NA__")
    vocab = {v: i + 1 for i, v in enumerate(sorted(vals.unique()))}
    cat_maps[c]       = vocab
    cat_vocab_sizes[c] = len(vocab) + 1
    train_df[c] = train_df[c].astype(str).map(vocab).fillna(0).astype(np.int64)
    val_df[c]   = val_df[c].astype(str).map(vocab).fillna(0).astype(np.int64)
    if c in test_pd.columns: test_pd[c] = test_pd[c].astype(str).map(vocab).fillna(0).astype(np.int64)

## Sequence Building
Converting continuous market segments into valid PyTorch `TensorDataset` configurations to feed into the LSTM module.

In [ ]:
log.info("Building sequences...")
model_feature_cols = numeric_cols_all + cat_feature_cols

static_feature_names   = [c for c in cat_feature_cols if c in model_feature_cols]
known_future_names     = [c for c in [TIME_COL, HORIZON_COL] if c in model_feature_cols and c not in static_feature_names]
# Dimensionality reduction handled intrinsically by Variable Selection Network (VSN) below over observed features
observed_feature_names = [c for c in model_feature_cols if c not in static_feature_names]

static_idx       = [model_feature_cols.index(c) for c in static_feature_names]
known_future_idx = [model_feature_cols.index(c) for c in known_future_names]
observed_idx     = [model_feature_cols.index(c) for c in observed_feature_names]

def make_sequences(df, seq_len, feature_cols, target_col, weight_col=None, group_cols=None, time_col="ts_index", max_samples=None):
    X, y, w = [], [], []
    if group_cols and all(c in df.columns for c in group_cols):
        groups   = df.groupby(group_cols, sort=False)
        iterable = (g.sort_values(time_col) for _, g in groups)
    else:
        iterable = [df.sort_values(time_col)]

    count = 0
    for gdf in iterable:
        vals = gdf[feature_cols].values.astype(np.float32)
        yt   = gdf[target_col].values.astype(np.float32)
        wt   = (gdf[weight_col].values.astype(np.float32) if weight_col else np.ones(len(gdf), dtype=np.float32))
        if len(gdf) <= seq_len: continue
        for i in range(seq_len, len(gdf)):
            if max_samples is None or count < max_samples:
                X.append(vals[i - seq_len:i])
                y.append(yt[i])
                w.append(wt[i])
            else:
                j = random.randint(0, count)
                if j < max_samples:
                    X[j] = vals[i - seq_len:i]
                    y[j] = yt[i]
                    w[j] = wt[i]
            count += 1
    return np.asarray(X), np.asarray(y), np.asarray(w)

X_train, y_train, w_train = make_sequences(train_df, SEQ_LEN, model_feature_cols, TARGET_COL, WEIGHT_COL, GROUP_COLS, TIME_COL, MAX_TRAIN_SEQ)
X_val, y_val, w_val = make_sequences(val_df, SEQ_LEN, model_feature_cols, TARGET_COL, WEIGHT_COL, GROUP_COLS, TIME_COL, MAX_VAL_SEQ)
log.info(f"X_train: {X_train.shape} | X_val: {X_val.shape}")

train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32), torch.tensor(w_train, dtype=torch.float32))
val_ds = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32), torch.tensor(w_val, dtype=torch.float32))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=0)

## Optimization Goals
Using `Weighted Mean Squared Error` against the observed weights of different assets.

In [ ]:
def weighted_mse(y_true, y_pred, w):
    # Using 'mean' perfectly stabilizes gradients across batches, 
    # instead of exploding if batch weight sum is abnormally low.
    return torch.mean(w * (y_true - y_pred) ** 2)

## Temporal Fusion Transformer (v2)
An optimized, low-parameter model designed not to overfit. Combines `MultiheadAttention`, `GLU`, `GRN`, and `VariableSelectionNetwork` modules.

In [ ]:
class GLU(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fc   = nn.Linear(dim, dim)
        self.gate = nn.Linear(dim, dim)
    def forward(self, x):
        return self.fc(x) * torch.sigmoid(self.gate(x))

class GRN(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim=None, dropout=0.1, context_dim=None):
        super().__init__()
        out_dim = out_dim or in_dim
        self.fc1        = nn.Linear(in_dim, hidden_dim)
        self.context_fc = nn.Linear(context_dim, hidden_dim, bias=False) if context_dim else None
        self.elu     = nn.ELU()
        self.fc2     = nn.Linear(hidden_dim, out_dim)
        self.dropout = nn.Dropout(dropout)
        self.glu     = GLU(out_dim)
        self.skip    = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()
        self.norm    = nn.LayerNorm(out_dim)
    def forward(self, x, context=None):
        h = self.fc1(x)
        if self.context_fc is not None and context is not None:
            while context.dim() < h.dim(): context = context.unsqueeze(1)
            h = h + self.context_fc(context)
        h = self.elu(h)
        h = self.dropout(h)
        h = self.fc2(h)
        h = self.glu(h)
        return self.norm(h + self.skip(x))

class VariableSelectionNetwork(nn.Module):
    def __init__(self, num_vars, hidden_dim, dropout=0.1, context_dim=None):
        super().__init__()
        self.num_vars = num_vars
        self.var_grns = nn.ModuleList([GRN(1, hidden_dim, hidden_dim, dropout) for _ in range(num_vars)])
        self.weight_grn = GRN(num_vars, hidden_dim, num_vars, dropout, context_dim=context_dim)
    def forward(self, x, context=None):
        is_temporal = x.dim() == 3
        if self.num_vars == 0:
            if is_temporal: return torch.zeros(x.size(0), x.size(1), 1, device=x.device), None
            return torch.zeros(x.size(0), 1, device=x.device), None
        if is_temporal:
            transformed = [self.var_grns[i](x[..., i:i+1]) for i in range(self.num_vars)]
            transformed = torch.stack(transformed, dim=-2)
            weights = self.weight_grn(x, context)
            weights = torch.softmax(weights, dim=-1).unsqueeze(-1)
            return (transformed * weights).sum(dim=-2), weights.squeeze(-1)
        else:
            transformed = [self.var_grns[i](x[:, i:i+1]) for i in range(self.num_vars)]
            transformed = torch.stack(transformed, dim=1)
            weights = self.weight_grn(x, context)
            weights = torch.softmax(weights, dim=-1).unsqueeze(-1)
            return (transformed * weights).sum(dim=1), weights.squeeze(-1)

class TFTv2(nn.Module):
    def __init__(self, static_idx, known_future_idx, observed_idx, cat_feature_indices, cat_vocab_sizes_list, hidden_dim=32, num_heads=2, dropout=0.35, embed_dim=8):
        super().__init__()
        self.cat_feature_indices = cat_feature_indices
        self.hidden_dim = hidden_dim
        self.cat_embeddings = nn.ModuleList([nn.Embedding(vs, embed_dim, padding_idx=0) for vs in cat_vocab_sizes_list])
        n_cont_observed  = len([i for i in observed_idx if i not in cat_feature_indices])
        static_input_dim = len(cat_feature_indices) * embed_dim
        self.static_fc = nn.Linear(static_input_dim, hidden_dim)
        self.static_context_grn = GRN(hidden_dim, hidden_dim, hidden_dim, dropout)
        self.observed_idx = observed_idx
        self.known_future_idx = known_future_idx
        self.observed_vsn = VariableSelectionNetwork(n_cont_observed, hidden_dim, dropout, context_dim=hidden_dim)
        self.known_vsn = VariableSelectionNetwork(len(known_future_idx), hidden_dim, dropout, context_dim=hidden_dim) if len(known_future_idx) > 0 else None
        self.lstm = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        self.post_lstm_grn = GRN(hidden_dim, hidden_dim, hidden_dim, dropout)
        self.static_enrichment = GRN(hidden_dim, hidden_dim, hidden_dim, dropout, context_dim=hidden_dim)
        self.attn = nn.MultiheadAttention(hidden_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.post_attn_grn = GRN(hidden_dim, hidden_dim, hidden_dim, dropout)
        self.position_grn  = GRN(hidden_dim, hidden_dim, hidden_dim, dropout)
        self.output_head = nn.Linear(hidden_dim, 1)
    def _causal_mask(self, t, device):
        return torch.triu(torch.ones(t, t, device=device), diagonal=1).bool()
    def forward(self, x):
        b, t, _ = x.shape
        if self.cat_feature_indices:
            cat_embeds = []
            for k, idx in enumerate(self.cat_feature_indices):
                cat_input = x[:, 0, idx].long().clamp(min=0)
                cat_embeds.append(self.cat_embeddings[k](cat_input))
            static_in  = torch.cat(cat_embeds, dim=-1)
            static_ctx = torch.relu(self.static_fc(static_in))
        else:
            static_ctx = torch.zeros(b, self.hidden_dim, device=x.device)
        static_ctx = self.static_context_grn(static_ctx)
        cont_idx = [i for i in self.observed_idx if i not in self.cat_feature_indices]
        x_obs = x[:, :, cont_idx]
        obs_repr, _ = self.observed_vsn(x_obs, context=static_ctx)
        if self.known_vsn is not None:
            x_known = x[:, :, self.known_future_idx]
            known_repr, _ = self.known_vsn(x_known, context=static_ctx)
            temporal_in = obs_repr + known_repr
        else:
            temporal_in = obs_repr
        lstm_out, _ = self.lstm(temporal_in)
        temporal = self.post_lstm_grn(lstm_out)
        temporal = self.static_enrichment(temporal, context=static_ctx)
        mask = self._causal_mask(t, x.device)
        attn_out, _ = self.attn(temporal, temporal, temporal, attn_mask=mask, need_weights=False)
        temporal = self.post_attn_grn(temporal + attn_out)
        temporal = self.position_grn(temporal)
        return self.output_head(temporal[:, -1, :]).squeeze(-1)

cat_indices_in_fv = [model_feature_cols.index(c) for c in cat_feature_cols]
cat_vsizes        = [cat_vocab_sizes[c] for c in cat_feature_cols]

model = TFTv2(
    static_idx           = static_idx,
    known_future_idx     = known_future_idx,
    observed_idx         = observed_idx,
    cat_feature_indices  = cat_indices_in_fv,
    cat_vocab_sizes_list = cat_vsizes,
    hidden_dim  = HIDDEN_DIM,
    num_heads   = NUM_HEADS,
    dropout     = DROPOUT,
    embed_dim   = EMBED_DIM,
).to(DEVICE)
log.info(f"Model initialized. Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Training Execution
Standard optimization loop against the validation boundaries. Checkpoints are automatically saved iteratively using `torch.save` over each epoch to allow ensemble creation later.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
WARMUP_EPOCHS = 3
def get_lr(epoch):
    if epoch < WARMUP_EPOCHS: return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1 + np.cos(np.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, get_lr)

train_losses, val_losses = [], []
best_val_loss = float('inf')

log.info("Starting Training...")
for epoch in range(1, EPOCHS + 1):
    t_ep = time.time()
    model.train()
    batch_losses = []
    
    for xb, yb, wb in train_loader:
        xb, yb, wb = xb.to(DEVICE), yb.to(DEVICE), wb.to(DEVICE)
        optimizer.zero_grad()
        pred = model(xb)
        loss = weighted_mse(yb, pred, wb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
        optimizer.step()
        batch_losses.append(loss.item())
        
    scheduler.step()
    ep_train_loss = np.mean(batch_losses)
    train_losses.append(ep_train_loss)

    # Validation Phase
    model.eval()
    val_batch_losses = []
    with torch.no_grad():
        for xb, yb, wb in val_loader:
            xb, yb, wb = xb.to(DEVICE), yb.to(DEVICE), wb.to(DEVICE)
            pred = model(xb)
            vloss = weighted_mse(yb, pred, wb)
            val_batch_losses.append(vloss.item())
    
    ep_val_loss = np.mean(val_batch_losses)
    val_losses.append(ep_val_loss)
    dt = time.time() - t_ep
    
    # Save Epoch Checkpoint
    checkpoint_path = OUTPUT_DIR / f'checkpoint_epoch_{epoch:02d}.pt'
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_loss': ep_val_loss,
    }, checkpoint_path)
    
    tag = " *" if ep_val_loss < best_val_loss else ""
    best_val_loss = min(best_val_loss, ep_val_loss)
    log.info(f"Epoch {epoch:02d} | Train Loss: {ep_train_loss:.6f} | Val Loss: {ep_val_loss:.6f} | Time: {dt:.1f}s | Saved: {checkpoint_path.name}{tag}")

log.info("Training Completed!")

## Validating Overfitting via Gradients
Generating epoch trajectory plot mapping temporal decay visually.

In [ ]:
# Plot Loss vs Epoch
plt.figure(figsize=(10, 6))
plt.plot(range(1, EPOCHS + 1), train_losses, label='Train Loss', marker='o', color='blue')
plt.plot(range(1, EPOCHS + 1), val_losses, label='Validation Loss', marker='s', color='orange')
plt.title('Training and Validation Loss vs. Epoch')
plt.xlabel('Epoch')
plt.ylabel('Weighted MSE Loss')
plt.legend()
plt.grid()
plot_path = OUTPUT_DIR / "loss_vs_epoch.png"
plt.savefig(plot_path)
plt.show()
log.info(f"Loss plot saved to {plot_path}")

## Sanity Testing (Local `test.parquet` Slice)
Ensures the model can process the testing dimensions and outputs valid values.

In [ ]:
log.info("Running Quick Sanity Check on test.parquet...")

# Load most recent model
checkpoints = sorted(list(OUTPUT_DIR.glob('checkpoint_epoch_*.pt')))
if len(checkpoints) > 0:
    best_chkpt = checkpoints[-1]
    log.info(f"Loading {best_chkpt.name} for inference...")
    checkpoint = torch.load(best_chkpt, map_location=DEVICE, weights_only=False)
        
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    # Grab the top rows from test_pd to form a few sequences
    test_slice = test_pd[model_feature_cols].head(SEQ_LEN * 4).values.astype(np.float32)
    
    X_test_sanity = []
    for i in range(SEQ_LEN, len(test_slice)+1):
        X_test_sanity.append(test_slice[i-SEQ_LEN:i])
        
    if len(X_test_sanity) > 0:
        t_test = torch.tensor(np.array(X_test_sanity), dtype=torch.float32).to(DEVICE)
        with torch.no_grad():
            preds = model(t_test)
        
        log.info(f"Sample Predictions on {len(preds)} sequences from test.parquet:")
        for idx, p in enumerate(preds.cpu().numpy()):
            log.info(f" Sequence {idx+1} Prediction -> {p:.6f}")
    else:
        log.warning("Not enough rows for sequence.")
